In [3]:
# This code simulates a single neuron of choice
from brian2 import *
import sys
sys.path.append('Neuron and Synapse Models')
from neuronModels import *
from ringAttractorVar import RingAttractor
sys.path.append('Tools')
from plottingTools import *


import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, Layout, interactive_output, FloatSlider, FloatText, Dropdown
from ipywidgets import interactive as interactive_ipyw

In [4]:
# Create sliders for parameters
timeStep_slider = FloatText(
    min=0.0, 
    max=1.0, 
    step=0.01,
    value=0.01, 
    description='Simulation Time Step:', 
    continuous_update=False, 
    readout_format='.2f',
    style={'description_width': '150px'},
)
tau_slider = FloatSlider(
    min=0.01, 
    max=10, 
    step=0.01, 
    value=5, 
    description='Tau Value:', 
    continuous_update=False,
)
Je_slider = FloatSlider(
    min=0.0,
    max=100.0,
    step=1.0,
    value=20.0,
    description='Je:',
    continuous_update=False,
)
Ji_slider = FloatSlider(
    min=0.0,
    max=100.0,
    step=1.0,
    value=0.5,
    description='Ji:',
    continuous_update=False,
)
Jei_slider = FloatSlider(
    min=0.0,
    max=100.0,
    step=1.0,
    value=0.0,
    description='Jei:',
    continuous_update=False,
)

widgets = {
    'timeStep': timeStep_slider,
    'tau_Val': tau_slider,
    'Je_Val': Je_slider,
    'Ji_Val': Ji_slider,
    'Jei_Val': Jei_slider
}

In [5]:
# Parameters
num_neurons = 50
Vth = -48*mV
V_reset = -80*mV
V_rest = -70*mV
neuron_eq = LIF_sim_eq

In [6]:
# Input Generation
# input_gen = SpikeGeneratorGroup(num_neurons, [np.random.randint(num_neurons)]*10, [0, 0, 0, 0, 0, 0]*10*ms)
trials = 3

In [7]:
def interactive_simulator(timeStep, tau_Val, Je_Val, Ji_Val, Jei_Val):
    
    input_gen = SpikeGeneratorGroup(num_neurons, [20]*trials, [i*10 for i in range(trials)]*ms)
    spike_monitor_input = SpikeMonitor(input_gen, name="spike_input")

    obj_network = RingAttractor(num_neurons, neuron_eq, tau_Val, Je_Val, Ji_Val, Jei_Val, Vth, V_rest, V_reset)
        
        
    connecting_input = Synapses(input_gen, obj_network.excitatory_neurons, model='W_input = (-Vth)/ohm : amp', name="input_synapses", on_pre="V_post += W_input*ohm")
    connecting_input.connect(j="i")


    obj_network.net.add(input_gen)
    obj_network.net.add(connecting_input)
    obj_network.net.add(spike_monitor_input)


    # Simulation Parameters
    #+---------------------------------------------------------------------------+
    defaultclock.dt = timeStep*ms
    duration = trials*10*ms
    obj_network.net.run(duration)
    
    
    plot_states = True
    if plot_states:
        fig, ax = plt.subplots(5,1, figsize=(15,10))
        ax[0].plot(spike_monitor_input.t/ms, spike_monitor_input.i, '.r', ms=3)
        ax[0].set_xlim(0, duration/ms)
        ax[0].set_ylim(0, obj_network.N)
        ax[0].set_xlabel('Time (ms)')
        ax[0].set_ylabel('Neuron idx')
        ax[0].set_title("Input Pattern")

        ax[1].plot(obj_network.spike_monitors.t/ms, obj_network.spike_monitors.i, '.k', ms=3)
        ax[1].set_xlim(0, duration/ms)
        ax[1].set_ylim(0, obj_network.N)
        ax[1].set_xlabel('Time (ms)')
        ax[1].set_ylabel('Neuron idx')
        ax[1].set_title("Excitatory neurons spiking activity")

        for n in range(obj_network.N):
            ax[2].plot(obj_network.state_monitors.t/ms , obj_network.state_monitors.V[n]/amp , label="%s" %n)
        
        ax[2].hlines(Vth/amp, 0, duration/ms, colors='r', linestyles='dashed', label='Threshold')
        ax[2].set_xlabel('Time (ms)')
        ax[2].set_ylabel('mV')
        ax[2].set_xlim([0, duration/ms])
        # ax[2].set_ylim([-90, 800])
        ax[2].set_title('Excitatory Neuron Membrane Potentials')
        # ax[2].legend()

        ax[3].plot(obj_network.inhibitory_spike_monitors.t/ms, obj_network.inhibitory_spike_monitors.i, '.k', ms=3)
        ax[3].set_xlabel('Time (ms)')
        ax[3].set_ylabel('mV')
        ax[3].set_xlim([0, duration/ms])
        ax[3].set_title('Inhibitory Neuron spiking activity')

        ax[4].plot(obj_network.inhibitory_state_monitors.t/ms, obj_network.inhibitory_state_monitors.V[0]/mV)
        ax[4].set_xlabel('Time (ms)')
        ax[4].set_ylabel('mV')
        ax[4].set_xlim([0, duration/ms])
        ax[4].set_title('Inhibitory Neuron Membrane Potential')

        plt.tight_layout()
        plt.show()

    # # TODO : Here I need a way to visualize the output of the network (which is inferred from the spiking of the excitatory neurons)

    # _, ax = plt.subplots(2,1,figsize=(20,15))
    # #lim_time = 50 ms
    # FR_neurons = [np.sum(obj_network.spike_monitors.i ==  neuron_idx)/(1e-3*(duration/ms)) for neuron_idx in range(num_neurons)]
    # peak = np.argmax(FR_neurons) #this is in neurons (i want to get it into degrees)
    # print("Correct direction %d" %(peak*3))
    # ax[0].plot(np.linspace(0,360,num_neurons), FR_neurons, "-r")
    # ax[0].set_xlabel("Neuron Idx")
    # ax[0].set_ylabel("FR [Hz]")
    # ax[0].set_title("Output Gaussian")
    # ax[0].set_xlim([0,360])

    # #I can also compute an histogram actually
    # output_spikes_hist = ax[1].hist(obj_network.spike_monitors.i, bins=num_neurons, range=(0, num_neurons), density=True) #/ (duration/ms)
    # mean_outputspikes = np.mean(obj_network.spike_monitors.i)
    # std_outputspikes = np.std(obj_network.spike_monitors.i)
    # x = np.arange(0,num_neurons)
    # fitted_gaussian = (1/(std_outputspikes * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mean_outputspikes) / std_outputspikes)**2)
    # ax[1].plot(np.arange(0,num_neurons), fitted_gaussian, label='Fitted Gaussian')
    # ax[1].set_title("Histogram and Fitted Gaussian")
    # ax[1].set_xlabel("Neuron Idx")
    # ax[1].set_ylabel("Density")
    # ax[1].set_xlim([0,num_neurons])
    # ax[1].set_xticks(np.linspace(0,num_neurons, 8), labels=["%.1f" %i for i in np.linspace(0,360,8)])
    # print("Fitted gaussian centered in %d, with a std equal to %.2f" %(np.argmax(fitted_gaussian)*3, std_outputspikes))

    # plt.show()

In [8]:
ui = VBox([timeStep_slider, tau_slider, Je_slider, Ji_slider, Jei_slider])
out = interactive_output(interactive_simulator, widgets)
display(HBox([out, ui], layout=Layout(align_items='center')))